# 07 · Exportar el paquete del póster web

El póster interactivo (`poster/streamlit_app.py`) se publica en **Streamlit Community Cloud**, que no tiene acceso al
workspace de Databricks. Este notebook toma las tablas Gold ya calculadas en la nube y arma un paquete liviano
(pocos MB) en `/Volumes/.../raw/poster/`:

| Archivo | Contenido |
|---|---|
| `h3_poster.parquet` | Un registro por hexágono H3: NDVI por año, pérdida oficial, perturbación, covariables, clusters LISA, Gi* y probabilidad del modelo |
| `capas/*.geojson` | Límite departamental, ríos, vía, ANP, zonas de amortiguamiento y poblados, recortados al área y simplificados |
| `resumen.json` | Cifras globales usadas en el póster |

`scripts/exportar_resultados.ps1` descarga esta carpeta a `poster/data/` del repositorio; al hacer *push*,
Streamlit Cloud actualiza el póster automáticamente.

In [ ]:
%pip install -q geopandas shapely h3
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [ ]:
import json, shutil, tempfile
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import box

OUT = Path(VOL) / "poster"
(OUT / "capas").mkdir(parents=True, exist_ok=True)

feat = spark.table(dbx.tbl("gold_h3_features")).toPandas().pipe(dbx.clean)
cl = spark.table(dbx.tbl("gold_h3_clusters")).toPandas().pipe(dbx.clean)
pr = (spark.table(dbx.tbl("gold_h3_prediccion")).toPandas().pipe(dbx.clean)
      .rename(columns={"n": "n_px_modelo"}))
df = (feat.merge(cl[["h3", "lisa", "gi_z", "hotspot"]], on="h3", how="left")
          .merge(pr[["h3", "prob_modelo", "frac_predicha", "n_px_modelo"]], on="h3", how="left"))

keep = ["h3", "lat", "lon", "departamento", "provincia", "distrito", "n_px_total",
        "frac_perdida_oficial", "tasa_perdida_oficial", "tasa_perdida_hansen", "frac_bosque_y0", "frac_bosque_2025",
        "riesgo_deforestacion", "d_ndvi", "d_pert", "dist_via_km", "dist_rio_km", "dist_cauce_km", "en_anp", "en_za",
        "lisa", "gi_z", "hotspot", "prob_modelo", "frac_predicha", "n_px_modelo"]
keep += [c for c in df.columns if c.startswith(("ndvi_", "pert_")) and c.split("_")[1].isdigit()]
keep = [c for c in dict.fromkeys(keep) if c in df.columns]
df = df[keep].copy()
for c in df.select_dtypes("float").columns:          # precisión suficiente para un mapa, archivo más liviano
    df[c] = df[c].astype("float32").round(4)
df.to_parquet(OUT / "h3_poster.parquet", index=False)
print(f"h3_poster.parquet: {len(df):,} hexágonos, {df.shape[1]} columnas, "
      f"{(OUT / 'h3_poster.parquet').stat().st_size / 1e6:.2f} MB")

In [ ]:
# Capas de contexto recortadas al área de estudio y simplificadas (≈50 m)
VEC = Path(VOL) / "bronze" / "vector"
caja = box(*C.AOI_BBOX_WGS84).buffer(0.03)
capas = {"departamentos": 0.0005, "rios": 0.0003, "vias": 0.0003, "areas_protegidas_sernanp": 0.0005,
         "zonas_amortiguamiento": 0.0005, "poblados": None}
for nombre, tol in capas.items():
    f = VEC / f"{nombre}.geojson"
    if not f.exists():
        print(f"  (no existe {f.name})")
        continue
    g = gpd.read_file(f).to_crs(4326)
    g = g[g.intersects(caja)].copy()
    if tol is not None:
        g["geometry"] = g.geometry.intersection(caja).simplify(tol, preserve_topology=True)
    g = g[~g.geometry.is_empty]
    cols = [c for c in ["name", "anp_nomb", "anp_cate", "departamento", "highway", "fclass"] if c in g.columns]
    g = g[cols + ["geometry"]]
    tmp = Path(tempfile.gettempdir()) / f"{nombre}.geojson"      # GDAL escribe en disco local y se copia al Volume
    g.to_file(tmp, driver="GeoJSON")
    shutil.copyfile(tmp, OUT / "capas" / f"{nombre}.geojson")
    print(f"  {nombre:28s} {len(g):5d} geometrías  {(OUT / 'capas' / f'{nombre}.geojson').stat().st_size / 1e3:8.1f} kB")

In [ ]:
# Resumen global (se muestra en el encabezado del póster)
HA_PX = 0.04
bos = df["frac_bosque_y0"] >= 0.5
res = {
    "hexagonos": int(len(df)),
    "hexagonos_bosque": int(bos.sum()),
    "perdida_oficial_ha": float((df["frac_perdida_oficial"] * df["n_px_total"]).sum() * HA_PX),
    "perdida_modelo_ha": float((df["prob_modelo"] * df["n_px_modelo"]).sum() * HA_PX),
    "frentes_lisa": int((df["lisa"] == "Pérdida-Pérdida (frente activo)").sum()),
    "calientes_99": int((df["hotspot"] == "Caliente 99%").sum()),
    "departamentos": sorted(df["departamento"].dropna().unique().tolist()) if "departamento" in df else [],
}
(OUT / "resumen.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(res, ensure_ascii=False, indent=2))

Siguiente paso, en la computadora local: `.\scripts\exportar_resultados.ps1` (copia `poster/` del Volume a
`poster/data/`), luego `git add`, `commit` y `push`. Streamlit Community Cloud vuelve a desplegar el póster solo.